# Semana 7 · Validación — Evaluación Fitosanitaria (Arañita Roja)
## Control de calidad y validación de datos

**Producto de la sesión:** dataset validado + resumen de reglas + incidencias + métricas + plan de acción.

> **Principio de trabajo:** validar no es volver a limpiar. Primero medimos, clasificamos y documentamos. No corregimos silenciosamente.

## 1. Archivo de entrada
Trabajaremos con el producto de la Semana 6: `fitosanitario_limpio.xlsx` (columnas en snake_case + flags de trazabilidad: `flag_duplicado_logico_corregido`, `flag_pct_hoja_imposible`, `flag_pct_postura_imposible`, `flag_grado_hoja_atipico`, `flag_grado_postura_atipico`).

In [ ]:
# Si estás en Google Colab, esto abre el selector de archivos.
# Sube fitosanitario_limpio.xlsx cuando te lo pida.
try:
    from google.colab import files
    uploaded = files.upload()
    ARCHIVO = next(iter(uploaded))
except ImportError:
    ARCHIVO = "fitosanitario_limpio.xlsx"

print("Archivo de trabajo:", ARCHIVO)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

df = pd.read_excel(ARCHIVO)

print("Dataset cargado correctamente.")
print("Dimensiones:", df.shape)
display(df.head(5))
print("\nColumnas disponibles:")
print(df.columns.tolist())

## 2. Contrato estructural del dataset
Antes de validar valores, comprobamos que existan las columnas indispensables. Si falta una, se detiene el proceso.

In [ ]:
COLUMNAS_NUCLEO = [
    "fecha", "campana", "modulo", "turno", "valvula", "plantas_evaluadas",
    "aranita_hojas_infestadas", "aranita_grado_hoja",
    "aranita_infestacion_postura", "aranita_grado_postura",
    "flag_duplicado_logico_corregido", "flag_pct_hoja_imposible", "flag_pct_postura_imposible",
    "flag_grado_hoja_atipico", "flag_grado_postura_atipico",
]

faltan_nucleo = [c for c in COLUMNAS_NUCLEO if c not in df.columns]
if faltan_nucleo:
    raise ValueError(f"El archivo no cumple el esquema esperado. Faltan columnas: {faltan_nucleo}")

print("✓ Esquema mínimo verificado.")

## 3. Preparación exclusiva para validar

In [ ]:
fecha_original = df["fecha"].copy()
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
fecha_no_convertida = fecha_original.notna() & df["fecha"].isna()
if fecha_no_convertida.any():
    raise ValueError(f"Hay {int(fecha_no_convertida.sum())} fechas no interpretables. Revise la Semana 6.")

for col in ["modulo", "turno", "valvula", "plantas_evaluadas",
            "aranita_hojas_infestadas", "aranita_grado_hoja",
            "aranita_infestacion_postura", "aranita_grado_postura"]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(f"La columna {col!r} debe ser numérica. Revise la Semana 6.")

FLAGS_SEMANA6 = ["flag_duplicado_logico_corregido", "flag_pct_hoja_imposible", "flag_pct_postura_imposible",
                  "flag_grado_hoja_atipico", "flag_grado_postura_atipico"]
faltan_flags = [c for c in FLAGS_SEMANA6 if c not in df.columns]
if faltan_flags:
    raise ValueError(f"Faltan banderas de trazabilidad de Semana 6: {faltan_flags}.")

for col in FLAGS_SEMANA6:
    df[col] = df[col].astype(bool)

print("✓ Fechas, variables numéricas y banderas verificadas.")

## 4. Política de severidad del caso

| Severidad | Umbral | Interpretación |
|---|---:|---|
| **CRÍTICA** | 100% | Una falla puede bloquear el uso del dataset. |
| **ALTA** | 98% | Requiere corrección prioritaria. |
| **MEDIA** | 95% | Puede permitir uso condicionado, pero debe documentarse. |
| **AVISO** | Sin umbral | Señal de revisión; no implica borrar automáticamente. |

## 5. Filtrado al alcance declarado del proyecto

Tu ficha declara como alcance oficial el periodo 01/03/2026–31/08/2026 (coincide con la ventana de clima disponible). Los registros fuera de esa ventana (evaluaciones de sep 2025 a feb 2026) no son un error de calidad — son datos reales que existen, pero quedan **fuera del alcance analítico** porque no hay clima con qué cruzarlos. Se documentan aparte, no se mezclan con las incidencias de calidad.

In [ ]:
INICIO_ALCANCE = pd.Timestamp("2026-03-01")
FIN_ALCANCE = pd.Timestamp("2026-08-31")

df["dentro_de_alcance"] = df["fecha"].between(INICIO_ALCANCE, FIN_ALCANCE)
print("Registros dentro del alcance declarado:", int(df["dentro_de_alcance"].sum()))
print("Registros fuera de alcance (sin clima con qué cruzar):", int((~df["dentro_de_alcance"]).sum()))

# Las reglas de calidad de las siguientes secciones se evalúan solo dentro del alcance.
df = df[df["dentro_de_alcance"]].reset_index(drop=True)
print("\nFilas que continúan al proceso de validación:", len(df))

## 6. Motor reusable de validación

In [ ]:
# Umbrales de aceptación por severidad (mismo criterio que Semana 7).
UMBRALES = {"CRÍTICA": 100.0, "ALTA": 98.0, "MEDIA": 95.0}

resumen_reglas = []
metadatos_reglas = {}

def _serie_booleana(valor, indice):
    if isinstance(valor, pd.Series):
        return valor.reindex(indice)
    return pd.Series(bool(valor), index=indice)

def registrar_regla(rule_id, dimension, campo, descripcion, severidad,
                     condicion_cumple, aplicable=None):
    indice = df.index
    cumple = _serie_booleana(condicion_cumple, indice).fillna(False).astype(bool)
    aplica = pd.Series(True, index=indice) if aplicable is None else _serie_booleana(aplicable, indice).fillna(False).astype(bool)

    incumple = aplica & (~cumple)
    col_error = f"err_{rule_id.lower()}"
    df[col_error] = incumple

    n_aplica = int(aplica.sum())
    n_incumple = int(incumple.sum())
    n_cumple = n_aplica - n_incumple
    porcentaje = round(100 * n_cumple / n_aplica, 2) if n_aplica > 0 else np.nan
    umbral = UMBRALES.get(severidad, np.nan)
    estado = "NO APLICA" if n_aplica == 0 else ("CUMPLE" if porcentaje >= umbral else "NO CUMPLE")

    resumen_reglas.append({
        "regla": rule_id, "dimension": dimension, "campo": campo,
        "descripcion": descripcion, "severidad": severidad, "umbral_pct": umbral,
        "aplicables": n_aplica, "cumplen": n_cumple, "incumplen": n_incumple,
        "cumplimiento_pct": porcentaje, "estado_regla": estado,
    })
    metadatos_reglas[rule_id] = {
        "dimension": dimension, "campo": campo, "descripcion": descripcion,
        "severidad": severidad, "col_error": col_error,
    }


## 7. Completitud

In [ ]:
registrar_regla("R01", "Completitud", "fecha", "Fecha obligatoria", "CRÍTICA", df["fecha"].notna())
registrar_regla("R02", "Completitud", "modulo", "Módulo obligatorio", "CRÍTICA", df["modulo"].notna())
registrar_regla("R03", "Completitud", "valvula", "Válvula obligatoria", "CRÍTICA", df["valvula"].notna())
registrar_regla("R04", "Completitud", "turno", "Turno obligatorio", "ALTA", df["turno"].notna())
registrar_regla("R05", "Completitud", "plantas_evaluadas", "Plantas evaluadas obligatorio", "ALTA", df["plantas_evaluadas"].notna())
print("✓ Reglas R01–R05 ejecutadas.")

## 8. Unicidad

In [ ]:
clave_negocio = ["fecha", "modulo", "valvula", "turno"]

registrar_regla("R06", "Unicidad", "fecha+modulo+valvula+turno",
    "La combinación fecha+módulo+válvula+turno no debe repetirse (los duplicados ya se resolvieron en la Semana 6)", "CRÍTICA",
    ~df.duplicated(subset=clave_negocio, keep=False))
print("✓ Regla R06 ejecutada.")

## 9. Validez: dominio y rango

In [ ]:
registrar_regla("R07", "Validez", "modulo", "Módulo debe pertenecer a {1,2,3,4,5,6}", "ALTA",
                 df["modulo"].isin({1,2,3,4,5,6}))

registrar_regla("R08", "Validez", "turno", "Turno debe pertenecer a {1,2,3,4,5,6}", "MEDIA",
                 df["turno"].isin({1,2,3,4,5,6}))

registrar_regla("R09", "Validez", "plantas_evaluadas", "Plantas evaluadas debe ser mayor a 0", "ALTA",
                 df["plantas_evaluadas"] > 0)

registrar_regla("R10", "Validez", "aranita_hojas_infestadas", "% hojas infestadas entre 0 y 100", "ALTA",
                 df["aranita_hojas_infestadas"].between(0, 100))

registrar_regla("R11", "Validez", "aranita_infestacion_postura", "% infestación postura entre 0 y 100", "ALTA",
                 df["aranita_infestacion_postura"].between(0, 100))

registrar_regla("R12", "Validez", "aranita_grado_hoja", "Grado hoja dentro de la escala 0-4", "MEDIA",
                 df["aranita_grado_hoja"].between(0, 4))

registrar_regla("R13", "Validez", "aranita_grado_postura", "Grado postura dentro de la escala 0-4", "MEDIA",
                 df["aranita_grado_postura"].between(0, 4))

print("✓ Reglas R07–R13 ejecutadas.")

## 10. Consistencia con la trazabilidad de la Semana 6
Igual que en el dataset de clima: no anulamos valores atípicos, solo los marcamos (un valor extremo aquí puede ser la señal real de un episodio de descontrol). La regla de consistencia verifica que **la bandera coincida con la condición**, no que el valor haya sido eliminado.

In [ ]:
registrar_regla("R14", "Consistencia", "aranita_hojas_infestadas + flag_pct_hoja_imposible",
    "flag_pct_hoja_imposible debe ser True solo si el % está fuera de [0,100]", "CRÍTICA",
    df["flag_pct_hoja_imposible"] == ~df["aranita_hojas_infestadas"].between(0, 100))

registrar_regla("R15", "Consistencia", "aranita_infestacion_postura + flag_pct_postura_imposible",
    "flag_pct_postura_imposible debe ser True solo si el % está fuera de [0,100]", "CRÍTICA",
    df["flag_pct_postura_imposible"] == ~df["aranita_infestacion_postura"].between(0, 100))

registrar_regla("R16", "Consistencia", "aranita_grado_hoja + flag_grado_hoja_atipico",
    "flag_grado_hoja_atipico debe ser True solo si el grado supera 4", "ALTA",
    df["flag_grado_hoja_atipico"] == (df["aranita_grado_hoja"] > 4))

registrar_regla("R17", "Consistencia", "aranita_grado_postura + flag_grado_postura_atipico",
    "flag_grado_postura_atipico debe ser True solo si el grado supera 4", "ALTA",
    df["flag_grado_postura_atipico"] == (df["aranita_grado_postura"] > 4))

print("✓ Reglas R14–R17 ejecutadas.")

## 11. Avisos de revisión
Un duplicado lógico o un valor alto de infestación no son automáticamente errores — en este proyecto, un pico alto es exactamente la señal de descontrol que queremos anticipar, no ruido a descartar.

In [ ]:
AVISOS = {
    "A01": {"dimension": "Revisión", "campo": "fecha+modulo+valvula+turno",
             "descripcion": "Evaluación que tuvo un duplicado lógico, ya resuelto en Semana 6 (se conservó la carga más reciente)", "severidad": "AVISO",
             "flag": "flag_duplicado_logico_corregido"},
    "A02": {"dimension": "Revisión", "campo": "aranita_grado_postura",
             "descripcion": "Grado postura fuera de la escala habitual (0-4); confirmar con el Ing. Kiyoshi", "severidad": "AVISO",
             "flag": "flag_grado_postura_atipico"},
}

resumen_reglas_df = pd.DataFrame(resumen_reglas)
display(resumen_reglas_df[["regla","dimension","campo","severidad","aplicables","incumplen","cumplimiento_pct","estado_regla"]])

## 12. Clasificación del estado de calidad por registro

In [ ]:
columnas_criticas_altas = [m["col_error"] for m in metadatos_reglas.values() if m["severidad"] in {"CRÍTICA","ALTA"}]
columnas_medias = [m["col_error"] for m in metadatos_reglas.values() if m["severidad"] == "MEDIA"]
columnas_avisos = [a["flag"] for a in AVISOS.values()]

tiene_falla_mayor = df[columnas_criticas_altas].any(axis=1)
tiene_observacion = df[columnas_medias].any(axis=1) | df[columnas_avisos].any(axis=1)

df["estado_calidad"] = np.select([tiene_falla_mayor, tiene_observacion], ["NO CONFORME","OBSERVADO"], default="ACEPTADO")

todas_columnas_error = [m["col_error"] for m in metadatos_reglas.values()]
df["n_incumplimientos"] = df[todas_columnas_error].sum(axis=1)
df["n_avisos"] = df[columnas_avisos].sum(axis=1)

display(df["estado_calidad"].value_counts(dropna=False).to_frame("registros"))

## 13. Tabla larga de incidencias

In [ ]:
def valor_observado(fila, campo):
    if "+" in campo:
        return ""
    return fila.get(campo, "")

incidencias = []
for idx, fila in df.iterrows():
    for rule_id, meta in metadatos_reglas.items():
        if bool(fila[meta["col_error"]]):
            incidencias.append({
                "fila": idx, "fecha": fila.get("fecha", pd.NA), "modulo": fila.get("modulo", pd.NA),
                "valvula": fila.get("valvula", pd.NA), "regla": rule_id, "dimension": meta["dimension"],
                "campo": meta["campo"], "severidad": meta["severidad"], "descripcion": meta["descripcion"],
                "valor_observado": valor_observado(fila, meta["campo"]), "estado_calidad": fila["estado_calidad"],
            })
    for aviso_id, aviso in AVISOS.items():
        if bool(fila.get(aviso["flag"], False)):
            incidencias.append({
                "fila": idx, "fecha": fila.get("fecha", pd.NA), "modulo": fila.get("modulo", pd.NA),
                "valvula": fila.get("valvula", pd.NA), "regla": aviso_id, "dimension": aviso["dimension"],
                "campo": aviso["campo"], "severidad": aviso["severidad"], "descripcion": aviso["descripcion"],
                "valor_observado": fila.get(aviso["campo"], ""), "estado_calidad": fila["estado_calidad"],
            })

incidencias_df = pd.DataFrame(incidencias)
print("Total incidencias:", len(incidencias_df))
display(incidencias_df.head(10))

## 14. Métricas globales por dimensión

In [ ]:
campos_requeridos = ["fecha", "modulo", "valvula", "turno", "plantas_evaluadas"]
completitud_pct = df[campos_requeridos].notna().sum().sum() / (len(df) * len(campos_requeridos)) * 100

clave = df[clave_negocio]
unicidad_pct = (~clave.duplicated(keep=False)).sum() / len(df) * 100 if len(df) else np.nan

def metrica_dimension(nombre):
    sub = resumen_reglas_df[resumen_reglas_df["dimension"] == nombre]
    total_aplica = sub["aplicables"].sum()
    total_cumple = sub["cumplen"].sum()
    return total_cumple / total_aplica * 100 if total_aplica else np.nan

metricas_calidad = pd.DataFrame({
    "metrica": ["Completitud campos obligatorios", "Unicidad clave de negocio", "Validez", "Consistencia"],
    "valor_pct": [completitud_pct, unicidad_pct, metrica_dimension("Validez"), metrica_dimension("Consistencia")],
})
display(metricas_calidad)

## 15. Plan de acción

In [ ]:
conteo_severidad = incidencias_df["severidad"].value_counts() if not incidencias_df.empty else pd.Series(dtype="int64")

plan_accion = pd.DataFrame([
    {"severidad": "CRÍTICA", "incidencias": int(conteo_severidad.get("CRÍTICA", 0)),
     "accion": "Contrastar con el formulario de campo original. Corregir si hay evidencia; si no, excluir y documentar.",
     "prioridad": "Inmediata"},
    {"severidad": "ALTA", "incidencias": int(conteo_severidad.get("ALTA", 0)),
     "accion": "Corregir prioritariamente o limitar el uso del registro hasta resolver.",
     "prioridad": "Alta"},
    {"severidad": "MEDIA", "incidencias": int(conteo_severidad.get("MEDIA", 0)),
     "accion": "Revisar, documentar y decidir si el análisis puede continuar con la limitación explícita.",
     "prioridad": "Programada"},
    {"severidad": "AVISO", "incidencias": int(conteo_severidad.get("AVISO", 0)),
     "accion": "Confirmar con el Ing. Kiyoshi (grado_postura atípico) o revisar contexto (duplicado lógico).",
     "prioridad": "Informativa"},
])
display(plan_accion)

## 16. Exportar evidencia reproducible

In [ ]:
ARCHIVO_VALIDADO = "fitosanitario_validado_semana7.csv"
ARCHIVO_INCIDENCIAS = "incidencias_validacion_fitosanitario.csv"
ARCHIVO_REPORTE = "reporte_validacion_fitosanitario.xlsx"

df.to_csv(ARCHIVO_VALIDADO, index=False, encoding="utf-8-sig")
incidencias_df.to_csv(ARCHIVO_INCIDENCIAS, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ARCHIVO_REPORTE, engine="openpyxl") as writer:
    resumen_reglas_df.to_excel(writer, sheet_name="Resumen_Reglas", index=False)
    metricas_calidad.to_excel(writer, sheet_name="Metricas_Calidad", index=False)
    incidencias_df.to_excel(writer, sheet_name="Incidencias", index=False)
    df.to_excel(writer, sheet_name="Dataset_Validado", index=False)
    plan_accion.to_excel(writer, sheet_name="Plan_Accion", index=False)

print("Archivos generados:")
print(" -", ARCHIVO_VALIDADO)
print(" -", ARCHIVO_INCIDENCIAS)
print(" -", ARCHIVO_REPORTE)

## 17. Decisión final del caso

In [ ]:
fallas_criticas = resumen_reglas_df[(resumen_reglas_df["severidad"]=="CRÍTICA") & (resumen_reglas_df["estado_regla"]=="NO CUMPLE")]
fallas_altas = resumen_reglas_df[(resumen_reglas_df["severidad"]=="ALTA") & (resumen_reglas_df["estado_regla"]=="NO CUMPLE")]

if len(fallas_criticas) > 0:
    estado_dataset = "REQUIERE CORRECCIÓN ANTES DEL ANÁLISIS"
elif len(fallas_altas) > 0:
    estado_dataset = "APTO SOLO CON CORRECCIONES PRIORITARIAS"
elif (df["estado_calidad"] == "OBSERVADO").any():
    estado_dataset = "APTO CON OBSERVACIONES DOCUMENTADAS"
else:
    estado_dataset = "APTO SEGÚN LAS REGLAS INTERNAS DEFINIDAS"

print("ESTADO DEL DATASET FITOSANITARIO:")
print(estado_dataset)